In [1]:
#Importing Libraries
import findspark
findspark.init()
print(findspark.find())

import os
import sys
import json
import time
import pymongo
import certifi
import shutil
import pandas as pd

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window as W

/opt/anaconda3/envs/pysparkenv/lib/python3.12/site-packages/pyspark


### 2.0. Instantiate Global Variables

In [2]:
# --------------------------------------------------------------------------------
# Specify MySQL Server Connection Information
# --------------------------------------------------------------------------------
mysql_args = {
    "host_name" : "localhost",
    "port" : "3306",
    "db_name" : "adventureworks",
    "conn_props" : {
        "user" : "root",
        "password" : "P0ssward",
        "driver" : "com.mysql.cj.jdbc.Driver"
    }
}

# --------------------------------------------------------------------------------
# Specify MongoDB Cluster Connection Information
# --------------------------------------------------------------------------------
mongodb_args = {
    "cluster_location" : "atlas", # "atlas"
    "user_name" : "gfusion333_db_user",
    "password" : "6YiBvJkTBJNdmJeP",
    "cluster_name" : "cluster0",
    "cluster_subnet" : "8xzjzzd",
    "db_name" : "adventureworks",
    "collection" : "",
    "null_column_threshold" : 0.5
}

# --------------------------------------------------------------------------------
# Specify Directory Structure for Source Data
# --------------------------------------------------------------------------------
base_dir = os.path.join(os.getcwd(), 'data')
data_dir = os.path.join(base_dir, 'adventureworks')
batch_dir = os.path.join(data_dir, 'batch')
stream_dir = os.path.join(data_dir, 'streaming')

purchase_orders_stream_dir = os.path.join(stream_dir, 'purchase_orders')

# --------------------------------------------------------------------------------
# Create Directory Structure for Data Lakehouse Files
# --------------------------------------------------------------------------------
dest_database = "adventureworks_dlh"
sql_warehouse_dir = os.path.abspath('spark-warehouse')
dest_database_dir = f"{dest_database}.db"
database_dir = os.path.join(sql_warehouse_dir, dest_database_dir)

purchase_orders_output_bronze = os.path.join(database_dir, 'fact_purchase_orders', 'bronze')
purchase_orders_output_silver = os.path.join(database_dir, 'fact_purchase_orders', 'silver')
purchase_orders_output_gold = os.path.join(database_dir, 'fact_purchase_orders', 'gold')

In [3]:
stream_dir

'/Users/gregory/Documents/GitHub/DS-2002/Projects/Final/data/adventureworks/streaming'

### 3.0. Define Global Functions

In [4]:
def get_file_info(path: str):
    file_sizes = []
    modification_times = []

    '''Fetch each item in the directory, and filter out any directories.'''
    items = os.listdir(path)
    files = sorted([item for item in items if os.path.isfile(os.path.join(path, item))])

    '''Populate lists with the Size and Last Modification DateTime for each file in the directory.'''
    for file in files:
        file_sizes.append(os.path.getsize(os.path.join(path, file)))
        modification_times.append(pd.to_datetime(os.path.getmtime(os.path.join(path, file)), unit='s'))

    data = list(zip(files, file_sizes, modification_times))
    column_names = ['name','size','modification_time']
    
    return pd.DataFrame(data=data, columns=column_names)


def wait_until_stream_is_ready(query, min_batches=1):
    while len(query.recentProgress) < min_batches:
        time.sleep(5)
        
    print(f"The stream has processed {len(query.recentProgress)} batchs")


def remove_directory_tree(path: str):
    '''If it exists, remove the entire contents of a directory structure at a given 'path' parameter's location.'''
    try:
        if os.path.exists(path):
            shutil.rmtree(path)
            return f"Directory '{path}' has been removed successfully."
        else:
            return f"Directory '{path}' does not exist."
            
    except Exception as e:
        return f"An error occurred: {e}"
        

def drop_null_columns(df, threshold):
    '''Drop Columns having a percentage of NULL values that exceeds the given 'threshold' parameter value.'''
    columns_with_nulls = [col for col in df.columns if df.filter(df[col].isNull()).count() / df.count() > threshold] 
    df_dropped = df.drop(*columns_with_nulls) 
    
    return df_dropped
    
    
def get_mysql_dataframe(spark_session, sql_query : str, **args):
    '''Create a JDBC URL to the MySQL Database'''
    jdbc_url = f"jdbc:mysql://{args['host_name']}:{args['port']}/{args['db_name']}"
    
    '''Invoke the spark.read.format("jdbc") function to query the database, and fill a DataFrame.'''
    dframe = spark_session.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("driver", args['conn_props']['driver']) \
    .option("user", args['conn_props']['user']) \
    .option("password", args['conn_props']['password']) \
    .option("query", sql_query) \
    .load()
    
    return dframe
    

def get_mongo_uri(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the 'cluster_location' parameter.")
        
    if args['cluster_location'] == "atlas":
        uri = f"mongodb+srv://{args['user_name']}:{args['password']}@"
        uri += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net/"
    else:
        uri = "mongodb://localhost:27017/"

    return uri


def get_spark_conf_args(spark_jars : list, **args):
    jars = ""
    for jar in spark_jars:
        jars += f"{jar}, "
    
    sparkConf_args = {
        "app_name" : "PySpark Adventureworks Data Lakehouse (Medallion Architecture)",
        "worker_threads" : f"local[{int(os.cpu_count()/2)}]",
        "shuffle_partitions" : int(os.cpu_count()),
        "mongo_uri" : get_mongo_uri(**args),
        "spark_jars" : jars[0:-2],
        "database_dir" : sql_warehouse_dir
    }
    
    return sparkConf_args
    

def get_spark_conf(**args):
    sparkConf = SparkConf().setAppName(args['app_name'])\
    .setMaster(args['worker_threads']) \
    .set('spark.driver.memory', '4g') \
    .set('spark.executor.memory', '2g') \
    .set('spark.jars', args['spark_jars']) \
    .set('spark.jars.packages', 'org.mongodb.spark:mongo-spark-connector_2.12:3.0.1') \
    .set('spark.mongodb.input.uri', args['mongo_uri']) \
    .set('spark.mongodb.output.uri', args['mongo_uri']) \
    .set('spark.sql.adaptive.enabled', 'false') \
    .set('spark.sql.debug.maxToStringFields', 35) \
    .set('spark.sql.shuffle.partitions', args['shuffle_partitions']) \
    .set('spark.sql.streaming.forceDeleteTempCheckpointLocation', 'true') \
    .set('spark.sql.streaming.schemaInference', 'true') \
    .set('spark.sql.warehouse.dir', args['database_dir']) \
    .set('spark.streaming.stopGracefullyOnShutdown', 'true')
    
    return sparkConf


def get_mongo_client(**args):
    '''Get MongoDB Client Connection'''
    mongo_uri = get_mongo_uri(**args)
    if args['cluster_location'] == "atlas":
        client = pymongo.MongoClient(mongo_uri, tlsCAFile=certifi.where())

    elif args['cluster_location'] == "local":
        client = pymongo.MongoClient(mongo_uri)
        
    else:
        raise Exception("A MongoDB Client could not be created.")

    return client
    
    
# TODO: Rewrite this to leverage PySpark?
def set_mongo_collections(mongo_client, db_name : str, data_directory : str, json_files : list):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()
    

def get_mongodb_dataframe(spark_session, **args):
    '''Query MongoDB, and create a DataFrame'''
    dframe = spark_session.read.format("com.mongodb.spark.sql.DefaultSource") \
        .option("database", args['db_name']) \
        .option("collection", args['collection']).load()

    '''Drop the '_id' index column to clean up the response.'''
    dframe = dframe.drop('_id')
    
    '''Call the drop_null_columns() function passing in the dataframe.'''
    dframe = drop_null_columns(dframe, args['null_column_threshold'])
    
    return dframe

### 4.0. Initialize Data Lakehouse Directory Structure
Remove the Data Lakehouse Database Directory Structure to Ensure Idempotency

In [5]:
remove_directory_tree(database_dir)

"Directory '/Users/gregory/Documents/GitHub/DS-2002/Projects/Final/spark-warehouse/adventureworks_dlh.db' has been removed successfully."

### 5.0. Create a New Spark Session

In [6]:
worker_threads = f"local[{int(os.cpu_count()/2)}]"

jars = []
mysql_spark_jar = '/Users/gregory/Documents/GitHub/DS-2002/05-Apache-Spark/05a-PySpark/mysql-connector-j-9.1.0/mysql-connector-j-9.1.0.jar'
mssql_spark_jar = os.path.join(os.getcwd(), "sqljdbc_12.8", "enu", "jars", "mssql-jdbc-12.8.1.jre11.jar")

jars.append(mysql_spark_jar)
#jars.append(mssql_spark_jar)

sparkConf_args = get_spark_conf_args(jars, **mongodb_args)

sparkConf = get_spark_conf(**sparkConf_args)
spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()
spark.sparkContext.setLogLevel("OFF")
spark

26/05/09 14:10:57 WARN Utils: Your hostname, Gregorys-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.5.208 instead (on interface en0)
26/05/09 14:10:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/gregory/.ivy2/cache
The jars for the packages stored in: /Users/gregory/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b498eaab-2c5e-441a-bf07-685e67155f2b;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.12;3.0.1 in central


:: loading settings :: url = jar:file:/opt/anaconda3/envs/pysparkenv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.mongodb#mongodb-driver-sync;4.0.5 in central
	found org.mongodb#bson;4.0.5 in central
	found org.mongodb#mongodb-driver-core;4.0.5 in central
:: resolution report :: resolve 99ms :: artifacts dl 4ms
	:: modules in use:
	org.mongodb#bson;4.0.5 from central in [default]
	org.mongodb#mongodb-driver-core;4.0.5 from central in [default]
	org.mongodb#mongodb-driver-sync;4.0.5 from central in [default]
	org.mongodb.spark#mongo-spark-connector_2.12;3.0.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   4   |   0   |   0   |   0   ||   4   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-b498eaab-2c5e-441a-bf07-685e6715

### 6.0. Create a New Metadata Database.

In [7]:
spark.sql(f"DROP DATABASE IF EXISTS {dest_database} CASCADE;")

sql_create_db = f"""
    CREATE DATABASE IF NOT EXISTS {dest_database}
    COMMENT 'DS-2002 Final'
    WITH DBPROPERTIES (contains_pii = true, purpose = 'DS-2002 Final');
"""
spark.sql(sql_create_db)

DataFrame[]

## Section II: Populate Dimensions by Ingesting "Cold-path" Reference Data 
### 1.0. Fetch Data from the File System
#### 1.1. Verify the location of the source data files on the file system

In [8]:
import os

search_root = "/Users/gregory"

for root, dirs, files in os.walk(search_root):
    for file in files:
        if file.startswith("mysql-connector") and file.endswith(".jar"):
            print(os.path.join(root, file))

/Users/gregory/Documents/GitHub/DS-2002/05-Apache-Spark/05a-PySpark/mysql-connector-j-9.1.0/mysql-connector-j-9.1.0.jar


In [9]:
get_file_info(batch_dir)

,name,size,modification_time
0,.DS_Store,6148,2026-05-09 17:34:07.367112398
1,adventureworks_customers.json,951569,2026-05-09 04:22:51.221525908
2,adventureworks_employees.csv,53785,2026-05-09 04:16:13.943468332
3,adventureworks_vendors.csv,11849,2026-05-09 04:15:00.702148914


#### 1.2. Populate the <span style="color:darkred">Employees Dimension</span>
##### 1.2.1. Use PySpark to Read data from a CSV file

In [10]:
employee_csv = os.path.join(batch_dir, 'adventureworks_employees.csv')
print(employee_csv)

df_dim_employees = spark.read.format('csv').options(header='true', inferSchema='true').load(employee_csv)
df_dim_employees.toPandas().head(2)

/Users/gregory/Documents/GitHub/DS-2002/Projects/Final/data/adventureworks/batch/adventureworks_employees.csv


,EmployeeID,NationalIDNumber,LoginID,ManagerID,FirstName,MiddleName,LastName,Title,EmailAddress,EmailPromotion,Phone,BirthDate,MaritalStatus,Gender,HireDate,SalariedFlag,VacationHours,SickLeaveHours,CurrentFlag
0,1,14417807,adventure-works\guy1,16,Guy,R,Gilbert,Production Technician - WC60,guy1@adventure-works.com,0,320-555-0195,1972-05-15,M,M,1996-07-31,0,21,30,1
1,2,253022876,adventure-works\kevin0,6,Kevin,F,Brown,Marketing Assistant,kevin0@adventure-works.com,2,150-555-0189,1977-06-03,S,M,1997-02-26,0,42,41,1


##### 1.2.2. Make Necessary Transformations to the New DataFrame

In [11]:
# ----------------------------------------------------------------------------------
# Add Primary Key column using SQL Windowing function: ROW_NUMBER() 
# ----------------------------------------------------------------------------------
df_dim_employees.createOrReplaceTempView("employees")
sql_employees = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY EmployeeID) AS employee_key
    FROM employees;
"""
df_dim_employees = spark.sql(sql_employees)

# ----------------------------------------------------------------------------------
# Reorder Columns and display the first two rows in a Pandas dataframe
# ----------------------------------------------------------------------------------
ordered_columns = ['employee_key', 'EmployeeID', 'NationalIDNumber', 'ManagerID'
                   , 'FirstName', 'MiddleName', 'LastName', 'Title', 'EmailAddress'
                   , 'Phone', 'BirthDate', 'MaritalStatus', 'Gender', 'HireDate'
                   , 'SalariedFlag', 'VacationHours', 'SickLeaveHours', 'CurrentFlag']

df_dim_employees = df_dim_employees[ordered_columns]
df_dim_employees.toPandas().head(2)

,employee_key,EmployeeID,NationalIDNumber,ManagerID,FirstName,MiddleName,LastName,Title,EmailAddress,Phone,BirthDate,MaritalStatus,Gender,HireDate,SalariedFlag,VacationHours,SickLeaveHours,CurrentFlag
0,1,1,14417807,16,Guy,R,Gilbert,Production Technician - WC60,guy1@adventure-works.com,320-555-0195,1972-05-15,M,M,1996-07-31,0,21,30,1
1,2,2,253022876,6,Kevin,F,Brown,Marketing Assistant,kevin0@adventure-works.com,150-555-0189,1977-06-03,S,M,1997-02-26,0,42,41,1


##### 1.2.3. Save as the <span style="color:darkred">dim_employees</span> table in the Data Lakehouse

In [12]:
df_dim_employees.write.saveAsTable(f"{dest_database}.dim_employees", mode="overwrite")

##### 1.2.4. Unit Test: Describe and Preview Table

In [13]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_employees;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_employees LIMIT 2").toPandas()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|        employee_key|      int|   NULL|
|          EmployeeID|      int|   NULL|
|    NationalIDNumber|      int|   NULL|
|           ManagerID|   string|   NULL|
|           FirstName|   string|   NULL|
|          MiddleName|   string|   NULL|
|            LastName|   string|   NULL|
|               Title|   string|   NULL|
|        EmailAddress|   string|   NULL|
|               Phone|   string|   NULL|
|           BirthDate|timestamp|   NULL|
|       MaritalStatus|   string|   NULL|
|              Gender|   string|   NULL|
|            HireDate|timestamp|   NULL|
|        SalariedFlag|      int|   NULL|
|       VacationHours|      int|   NULL|
|      SickLeaveHours|      int|   NULL|
|         CurrentFlag|      int|   NULL|
|                    |         |       |
|# Detailed Table ...|         |       |
+--------------------+---------+-------+
only showing top

,employee_key,EmployeeID,NationalIDNumber,ManagerID,FirstName,MiddleName,LastName,Title,EmailAddress,Phone,BirthDate,MaritalStatus,Gender,HireDate,SalariedFlag,VacationHours,SickLeaveHours,CurrentFlag
0,1,1,14417807,16,Guy,R,Gilbert,Production Technician - WC60,guy1@adventure-works.com,320-555-0195,1972-05-15,M,M,1996-07-31,0,21,30,1
1,2,2,253022876,6,Kevin,F,Brown,Marketing Assistant,kevin0@adventure-works.com,150-555-0189,1977-06-03,S,M,1997-02-26,0,42,41,1


#### 1.3. Populate the <span style="color:darkred">Vendors Dimension</span>
##### 1.3.1. Use PySpark to Read Data from a CSV File

In [14]:
# 1). Get a reference to the 'adventureworks_vendors.csv' file.
vendors_csv = os.path.join(batch_dir, 'adventureworks_vendors.csv')
print(vendors_csv)

# 2). Use Spark to read the CSV file data into the 'df_dim_shippers' variable.
#     Remember to specify that the first row contains column names (header), and to infer the schema.
df_dim_vendors = spark.read.format('csv').options(header='true', inferSchema='true').load(vendors_csv)

# 3). Unit Test: Convert the spark dataframe to a Pandas dataframe, and display the first two rows.
df_dim_vendors.toPandas().head(2)

/Users/gregory/Documents/GitHub/DS-2002/Projects/Final/data/adventureworks/batch/adventureworks_vendors.csv


,VendorID,AccountNumber,Name,CreditRating,PreferredVendorStatus,ActiveFlag,AddressType,AddressLine1,AddressLine2,City,StateProvinceCode,State_Province,PostalCode
0,1,INTERNAT0001,International,1,1,1,Main Office,683 Larch Ct.,NULL,Salt Lake City,UT,Utah,84101
1,2,ELECTRON0002,Electronic Bike Repair & Supplies,1,1,1,Main Office,8547 Catherine Way,NULL,Tacoma,WA,Washington,98403


##### 1.3.2 Make Necessary Transformations to the New DataFrame

In [16]:
# ----------------------------------------------------------------------------------
# Rename Name column to vendor_name
# ----------------------------------------------------------------------------------
df_dim_vendors = df_dim_vendors.withColumnRenamed("Name", "vendor_name")

# ----------------------------------------------------------------------------------
# Add Primary Key column using SQL Windowing function: ROW_NUMBER() 
# ----------------------------------------------------------------------------------
df_dim_vendors.createOrReplaceTempView("vendors")
sql_vendors = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY VendorID) AS vendor_key
    FROM vendors;
"""
df_dim_vendors = spark.sql(sql_vendors)

# ----------------------------------------------------------------------------------
# Reorder Columns and display the first two rows in a Pandas dataframe
# ----------------------------------------------------------------------------------
ordered_columns = ['vendor_key', 'VendorID', 'AccountNumber', 'vendor_name', 'CreditRating'
                   , 'PreferredVendorStatus', 'ActiveFlag', 'AddressType', 'AddressLine1'
                   , 'AddressLine2', 'City', 'StateProvinceCode', 'State_Province', 'PostalCode']

df_dim_vendors = df_dim_vendors[ordered_columns]
df_dim_vendors.toPandas().head(2)

,vendor_key,VendorID,AccountNumber,vendor_name,CreditRating,PreferredVendorStatus,ActiveFlag,AddressType,AddressLine1,AddressLine2,City,StateProvinceCode,State_Province,PostalCode
0,1,1,INTERNAT0001,International,1,1,1,Main Office,683 Larch Ct.,NULL,Salt Lake City,UT,Utah,84101
1,2,2,ELECTRON0002,Electronic Bike Repair & Supplies,1,1,1,Main Office,8547 Catherine Way,NULL,Tacoma,WA,Washington,98403


##### 1.3.3. Save as the <span style="color:darkred">dim_vendors</span> table in the Data Lakehouse

In [17]:
df_dim_vendors.write.saveAsTable(f"{dest_database}.dim_vendors", mode="overwrite")

##### 1.3.4. Unit Test: Describe and Preview Table

In [18]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_vendors;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_vendors LIMIT 2").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|          vendor_key|                 int|   NULL|
|            VendorID|                 int|   NULL|
|       AccountNumber|              string|   NULL|
|         vendor_name|              string|   NULL|
|        CreditRating|                 int|   NULL|
|PreferredVendorSt...|                 int|   NULL|
|          ActiveFlag|                 int|   NULL|
|         AddressType|              string|   NULL|
|        AddressLine1|              string|   NULL|
|        AddressLine2|              string|   NULL|
|                City|              string|   NULL|
|   StateProvinceCode|              string|   NULL|
|      State_Province|              string|   NULL|
|          PostalCode|                 int|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|           

,vendor_key,VendorID,AccountNumber,vendor_name,CreditRating,PreferredVendorStatus,ActiveFlag,AddressType,AddressLine1,AddressLine2,City,StateProvinceCode,State_Province,PostalCode
0,1,1,INTERNAT0001,International,1,1,1,Main Office,683 Larch Ct.,NULL,Salt Lake City,UT,Utah,84101
1,2,2,ELECTRON0002,Electronic Bike Repair & Supplies,1,1,1,Main Office,8547 Catherine Way,NULL,Tacoma,WA,Washington,98403


### 2.0. Fetch Reference Data from a MongoDB Atlas Database
#### 2.1. Create a New MongoDB Database, and Load Each JSON File into a New MongoDB Collection
**NOTE:** The following cell **can** be run more than once because the **set_mongo_collection()** function **is** idempotent.

In [19]:
client = get_mongo_client(**mongodb_args)

json_files = {"customers" : "adventureworks_customers.json"}

set_mongo_collections(client, mongodb_args["db_name"], batch_dir, json_files) 

#### 2.2. Populate the <span style="color:darkred">Customers Dimension</span>
##### 2.2.1. Fetch Data from the New MongoDB <span style="color:darkred">Customers</span> Collection

In [20]:
mongodb_args["collection"] = "customers"

df_dim_customers = get_mongodb_dataframe(spark, **mongodb_args)
df_dim_customers.toPandas().head(2)

,AccountNumber,AddressLine1,AddressType,City,CountryRegionCode,Country_Region,CustomerID,CustomerType,IsOnlyStateProvinceFlag,PostalCode,Sales Territory,Sales Territory Group,StateProvinceCode,State_Province
0,AW00000001,2251 Elliot Avenue,Main Office,Seattle,US,United States,1,S,0,98104,Northwest,North America,WA,Washington
1,AW00000002,7943 Walnut Ave,Shipping,Renton,US,United States,2,S,0,98055,Northwest,North America,WA,Washington


##### 2.2.2. Make Necessary Transformations to the New Dataframe

In [21]:
# ----------------------------------------------------------------------------------
# Add Primary Key column using the SQL Windowing function: ROW_NUMBER() 
# ----------------------------------------------------------------------------------
df_dim_customers.createOrReplaceTempView("customers")
sql_customers = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY CustomerID) AS customer_key
    FROM customers;
"""
df_dim_customers = spark.sql(sql_customers)

# ----------------------------------------------------------------------------------
# Reorder Columns and display the first two rows in a Pandas dataframe
# ----------------------------------------------------------------------------------
ordered_columns = ['customer_key', 'CustomerID', 'AccountNumber', 'CustomerType', 'AddressType', 'AddressLine1', 'City', 'State_Province', 'StateProvinceCode', 'PostalCode', 'CountryRegionCode', 'Country_Region', 'Sales Territory', 'Sales Territory Group', 'IsOnlyStateProvinceFlag']

df_dim_customers = df_dim_customers[ordered_columns]
df_dim_customers.toPandas().head(2)

,customer_key,CustomerID,AccountNumber,CustomerType,AddressType,AddressLine1,City,State_Province,StateProvinceCode,PostalCode,CountryRegionCode,Country_Region,Sales Territory,Sales Territory Group,IsOnlyStateProvinceFlag
0,1,1,AW00000001,S,Main Office,2251 Elliot Avenue,Seattle,Washington,WA,98104,US,United States,Northwest,North America,0
1,2,2,AW00000002,S,Shipping,7943 Walnut Ave,Renton,Washington,WA,98055,US,United States,Northwest,North America,0


##### 2.2.3. Save as the <span style="color:darkred">dim_customers</span> table in the Data lakehouse

In [22]:
df_dim_customers.write.saveAsTable(f"{dest_database}.dim_customers", mode="overwrite")

##### 2.2.4. Unit Test: Describe and Preview Table

In [23]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_customers;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_customers LIMIT 2").toPandas()

+--------------------+------------------+-------+
|            col_name|         data_type|comment|
+--------------------+------------------+-------+
|        customer_key|               int|   NULL|
|          CustomerID|               int|   NULL|
|       AccountNumber|            string|   NULL|
|        CustomerType|            string|   NULL|
|         AddressType|            string|   NULL|
|        AddressLine1|            string|   NULL|
|                City|            string|   NULL|
|      State_Province|            string|   NULL|
|   StateProvinceCode|            string|   NULL|
|          PostalCode|            string|   NULL|
|   CountryRegionCode|            string|   NULL|
|      Country_Region|            string|   NULL|
|     Sales Territory|            string|   NULL|
|Sales Territory G...|            string|   NULL|
|IsOnlyStateProvin...|            string|   NULL|
|                    |                  |       |
|# Detailed Table ...|                  |       |


,customer_key,CustomerID,AccountNumber,CustomerType,AddressType,AddressLine1,City,State_Province,StateProvinceCode,PostalCode,CountryRegionCode,Country_Region,Sales Territory,Sales Territory Group,IsOnlyStateProvinceFlag
0,1,1,AW00000001,S,Main Office,2251 Elliot Avenue,Seattle,Washington,WA,98104,US,United States,Northwest,North America,0
1,2,2,AW00000002,S,Shipping,7943 Walnut Ave,Renton,Washington,WA,98055,US,United States,Northwest,North America,0


### 3.0. Fetch Reference Data from a MySQL Database
#### 3.1. Populate the <span style="color:darkred">Date Dimension</span>
##### 3.1.1 Fetch data from the <span style="color:darkred">dim_date</span> table in MySQL

In [24]:
sql_dim_date = f"SELECT * FROM {mysql_args['db_name']}.dim_date"
df_dim_date = get_mysql_dataframe(spark, sql_dim_date, **mysql_args)

##### 3.1.2. Save as the <span style="color:darkred">dim_date</span> table in the Data Lakehouse

In [25]:
df_dim_date.write.saveAsTable(f"{dest_database}.dim_date", mode="overwrite")

##### 3.1.3. Unit Test: Describe and Preview Table

In [26]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_date;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_date LIMIT 2").toPandas()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|            date_key|      int|   NULL|
|           full_date|     date|   NULL|
|           date_name| char(11)|   NULL|
|        date_name_us| char(11)|   NULL|
|        date_name_eu| char(11)|   NULL|
|         day_of_week|  tinyint|   NULL|
|    day_name_of_week| char(10)|   NULL|
|        day_of_month|  tinyint|   NULL|
|         day_of_year|      int|   NULL|
|     weekday_weekend| char(10)|   NULL|
|        week_of_year|  tinyint|   NULL|
|          month_name| char(10)|   NULL|
|       month_of_year|  tinyint|   NULL|
|is_last_day_of_month|  char(1)|   NULL|
|    calendar_quarter|  tinyint|   NULL|
|       calendar_year|      int|   NULL|
| calendar_year_month| char(10)|   NULL|
|   calendar_year_qtr| char(10)|   NULL|
|fiscal_month_of_year|  tinyint|   NULL|
|      fiscal_quarter|  tinyint|   NULL|
+--------------------+---------+-------+
only showing top

,date_key,full_date,date_name,date_name_us,date_name_eu,day_of_week,day_name_of_week,day_of_month,day_of_year,weekday_weekend,...,is_last_day_of_month,calendar_quarter,calendar_year,calendar_year_month,calendar_year_qtr,fiscal_month_of_year,fiscal_quarter,fiscal_year,fiscal_year_month,fiscal_year_qtr
0,20000101,2000-01-01,2000/01/01,01/01/2000,01/01/2000,7,Saturday,1,1,Weekend,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3
1,20000102,2000-01-02,2000/01/02,01/02/2000,02/01/2000,1,Sunday,2,2,Weekend,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3


#### 3.2. Populate the <span style="color:darkred">Product Dimension</span>
##### 3.2.1. Fetch data from the <span style="color:darkred">Products</span> table in MySQL

In [28]:
# ----------------------------------------------------------------------------------
# Add Primary Key column using the SQL Windowing function: ROW_NUMBER() 
# ----------------------------------------------------------------------------------
sql_products = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY ProductID) AS product_key
    FROM {mysql_args['db_name']}.dim_products_vw
"""
df_dim_products = get_mysql_dataframe(spark, sql_products, **mysql_args)

##### 3.2.2. Perform any Necessary Transformations

In [30]:
# ----------------------------------------------------------------------------------
# Reorder Columns and display the first two rows in a Pandas dataframe
# ----------------------------------------------------------------------------------
ordered_columns = ['product_key', 'ProductID', 'Name', 'ProductNumber'
                   , 'MakeFlag', 'FinishedGoodsFlag', 'Color', 'SafetyStockLevel', 'ReorderPoint'
                   , 'StandardCost', 'ListPrice', 'Size', 'DaysToManufacture', 'Class'
                  , 'SellStartDate', 'SellEndDate']

df_dim_products = df_dim_products[ordered_columns]
df_dim_products = df_dim_products.withColumnRenamed("Name", "product_name")
df_dim_products.toPandas().head(2)

,product_key,ProductID,product_name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,DaysToManufacture,Class,SellStartDate,SellEndDate
0,1,1,Adjustable Race,AR-5381,False,False,None,1000,750,0.0,0.0,None,0,None,1998-06-01,NaT
1,2,2,Bearing Ball,BA-8327,False,False,None,1000,750,0.0,0.0,None,0,None,1998-06-01,NaT


##### 3.2.3. Save as the <span style="color:darkred">dim_products</span> table in the Data Lakehouse

In [31]:
df_dim_products.write.saveAsTable(f"{dest_database}.dim_products", mode="overwrite")

##### 3.2.4. Unit Test: Describe and Preview Table

In [32]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_products;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_products LIMIT 2").toPandas()

+--------------------+------------------+-------+
|            col_name|         data_type|comment|
+--------------------+------------------+-------+
|         product_key|     decimal(20,0)|   NULL|
|           ProductID|               int|   NULL|
|        product_name|       varchar(50)|   NULL|
|       ProductNumber|       varchar(25)|   NULL|
|            MakeFlag|           boolean|   NULL|
|   FinishedGoodsFlag|           boolean|   NULL|
|               Color|       varchar(15)|   NULL|
|    SafetyStockLevel|               int|   NULL|
|        ReorderPoint|               int|   NULL|
|        StandardCost|            double|   NULL|
|           ListPrice|            double|   NULL|
|                Size|        varchar(5)|   NULL|
|   DaysToManufacture|               int|   NULL|
|               Class|        varchar(2)|   NULL|
|       SellStartDate|         timestamp|   NULL|
|         SellEndDate|         timestamp|   NULL|
|                    |                  |       |


,product_key,ProductID,product_name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,DaysToManufacture,Class,SellStartDate,SellEndDate
0,1,1,Adjustable Race,AR-5381,False,False,None,1000,750,0.0,0.0,None,0,None,1998-06-01,NaT
1,2,2,Bearing Ball,BA-8327,False,False,None,1000,750,0.0,0.0,None,0,None,1998-06-01,NaT


### 4.0. Verify Dimension Tables

In [33]:
spark.sql(f"USE {dest_database};")
spark.sql("SHOW TABLES").toPandas()

,namespace,tableName,isTemporary
0,adventureworks_dlh,dim_customers,False
1,adventureworks_dlh,dim_date,False
2,adventureworks_dlh,dim_employees,False
3,adventureworks_dlh,dim_products,False
4,adventureworks_dlh,dim_vendors,False
5,,customers,True
6,,employees,True
7,,vendors,True


## Section III: Integrate Reference Data with Real-Time Data

### 8.0. Use PySpark Structured Streaming to Process (Hot Path) <span style="color:darkred">Purchase Orders</span> Fact Data
#### 8.1. Verify the location of the source data files on the file system

In [34]:
get_file_info(purchase_orders_stream_dir)

,name,size,modification_time
0,adventureworks_purchase_orders_01.json,1765391,2026-05-09 17:26:46.977617741
1,adventureworks_purchase_orders_02.json,1769104,2026-05-09 17:27:13.409617662
2,adventureworks_purchase_orders_03.json,1644062,2026-05-09 17:27:45.819882393


#### 8.2. Create the Bronze Layer: Stage <span style="color:darkred">Purchase Orders Fact table</span> Data
##### 8.2.1. Read "Raw" JSON file data into a Stream

In [35]:
df_purchase_orders_bronze = (
    spark.readStream \
    .option("schemaLocation", purchase_orders_output_bronze) \
    .option("maxFilesPerTrigger", 1) \
    .option("multiLine", "true") \
    .json(purchase_orders_stream_dir))
    
df_purchase_orders_bronze.isStreaming

True

##### 8.2.2. Write the Streaming Data to a Parquet file

In [36]:
purchase_orders_checkpoint_bronze = os.path.join(purchase_orders_output_bronze, '_checkpoint')

purchase_orders_bronze_query = (
    df_purchase_orders_bronze
    # TODO: Add Current Timestamp and Input Filename columns for Traceability
    # TODO: writeStream to 'purchase_orders_output_bronze' in 'append' mode
    .withColumn("receipt_time", current_timestamp())
    .withColumn("source_file", input_file_name())
    
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .queryName("purchase_orders_bronze")
    .trigger(availableNow = True) \
    .option("checkpointLocation", purchase_orders_checkpoint_bronze) \
    .option("compression", "snappy") \
    .start(purchase_orders_output_bronze)
)

##### 8.2.3. Unit Test: Implement Query Monitoring

In [37]:
print(f"Query ID: {purchase_orders_bronze_query.id}")
print(f"Query Name: {purchase_orders_bronze_query.name}")
print(f"Query Status: {purchase_orders_bronze_query.status}")

Query ID: 518fa84c-d164-4c21-b8e2-394c66863461
Query Name: purchase_orders_bronze
Query Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [38]:
purchase_orders_bronze_query.awaitTermination()

#### 8.3. Create the Silver Layer: Integrate "Cold-path" Data & Make Transformations
##### 8.3.1. Prepare Role-Playing Dimension Primary and Business Keys

In [39]:
df_dim_ship_date = df_dim_date.select(col("date_key").alias("ship_date_key"), col("full_date").alias("ship_full_date"))
#TODO: Copy df_dim_date and rename 'date_key' and 'full_date' columns.
df_dim_order_date = df_dim_date.select(col("date_key").alias("order_date_key"), col("full_date").alias("order_full_date"))
#TODO: Copy df_dim_date and rename 'date_key' and 'full_date' columns.
df_dim_due_date = df_dim_date.select(col("date_key").alias("due_date_key"), col("full_date").alias("due_full_date"))
#TODO: Copy df_dim_date and rename 'date_key' and 'full_date' columns.

##### 8.3.2. Define Silver Query to Join Streaming with Batch Data

In [40]:
df_purchase_orders_silver = spark.readStream.format("parquet").load(purchase_orders_output_bronze) \
    .join(df_dim_products, "ProductID") \
    .join(df_dim_vendors, "VendorID") \
    .join(df_dim_employees, "EmployeeID") \
    .join(df_dim_ship_date, df_dim_ship_date.ship_full_date.cast(DateType()) == col("ShipDate").cast(DateType()), "inner") \
    .join(df_dim_order_date, df_dim_order_date.order_full_date.cast(DateType()) == col("OrderDate").cast(DateType()), "inner") \
    .join(df_dim_due_date, df_dim_due_date.due_full_date.cast(DateType()) == col("DueDate").cast(DateType()), "left_outer") \
    .select(col("PurchaseOrderID"), \
            df_dim_products.product_key.cast(IntegerType()), \
            df_dim_vendors.vendor_key.cast(IntegerType()), \
            df_dim_employees.employee_key.cast(IntegerType()), \
            df_dim_ship_date.ship_date_key.cast(LongType()), \
            df_dim_order_date.order_date_key.cast(LongType()), \
            df_dim_due_date.due_date_key.cast(LongType()), \
            col("RevisionNumber"), \
            col("Status"), \
            col("OrderQty"), \
            col("UnitPrice"), \
            col("LineTotal"), \
            col("ShipMethod"), \
            col("ShipBase"), \
            col("ShipRate"), \
            col("SubTotal"), \
            col("TaxAmt"), \
            col("Freight"), \
            col("TotalDue"), \
            col("ReceivedQty"), \
            col("RejectedQty"), \
            col("StockedQty") \
           )
    # join dimension tables and .select() the appropriate columns from the 'purchase orders bronze' stream


In [41]:
df_purchase_orders_silver.isStreaming

True

In [42]:
df_purchase_orders_silver.printSchema()

root
 |-- PurchaseOrderID: long (nullable = true)
 |-- product_key: integer (nullable = true)
 |-- vendor_key: integer (nullable = false)
 |-- employee_key: integer (nullable = false)
 |-- ship_date_key: long (nullable = true)
 |-- order_date_key: long (nullable = true)
 |-- due_date_key: long (nullable = true)
 |-- RevisionNumber: long (nullable = true)
 |-- Status: long (nullable = true)
 |-- OrderQty: long (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- LineTotal: double (nullable = true)
 |-- ShipMethod: string (nullable = true)
 |-- ShipBase: double (nullable = true)
 |-- ShipRate: double (nullable = true)
 |-- SubTotal: double (nullable = true)
 |-- TaxAmt: double (nullable = true)
 |-- Freight: double (nullable = true)
 |-- TotalDue: double (nullable = true)
 |-- ReceivedQty: double (nullable = true)
 |-- RejectedQty: double (nullable = true)
 |-- StockedQty: double (nullable = true)



##### 8.3.3. Write the Transformed Streaming data to the Data Lakehouse

In [43]:
purchase_orders_checkpoint_silver = os.path.join(purchase_orders_output_silver, '_checkpoint')

purchase_orders_silver_query = (
    df_purchase_orders_silver.writeStream \
    # TODO: writeStream, in 'parquet' format, to 'purchase_orders_output_silver' in 'append' mode
    .format("parquet") \
    .outputMode("append") \
    .queryName("purchase_orders_silver")
    .trigger(availableNow = True) \
    .option("checkpointLocation", purchase_orders_checkpoint_silver) \
    .option("compression", "snappy") \
    .start(purchase_orders_output_silver)
)

##### 8.3.4. Unit Test: Implement Query Monitoring

In [44]:
print(f"Query ID: {purchase_orders_silver_query.id}")
print(f"Query Name: {purchase_orders_silver_query.name}")
print(f"Query Status: {purchase_orders_silver_query.status}")

Query ID: 9e022cf8-ce29-4935-b595-734edb2e4b05
Query Name: purchase_orders_silver
Query Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [45]:
purchase_orders_silver_query.awaitTermination()

#### 8.4. Create Gold Layer: Perform Aggregations
##### 8.4.1. Define a Query to Create a Business Report
Create a new Gold table using the PySpark API. The table should include the Suppliers' Company Name, the Product Name, the Total Quantity, Total Unit Cost, and Total List Price for all the purchase orders placed per Supplier for each Product.

In [46]:
df_fact_pos_products_per_supplier_gold = spark.readStream.format("parquet").load(purchase_orders_output_silver) \
.join(df_dim_products, "product_key") \
.join(df_dim_vendors, "vendor_key") \
.groupBy("vendor_name", "product_Name") \
.agg(sum("OrderQty").alias("total_quantity"),
    sum("TotalDue").alias("total_cost")) \
.orderBy(desc("total_cost"))
# .join to the 'df_dim_products' dimension
# .join to the 'df_dim_suppliers' dimension
# .groupBy 'company' and 'product_name'
# sum 'po_detail_quantity' as 'Total Quantity'
# sum 'po_detail_unit_cost' as 'Total Unit Cost'
# sum 'list_price' as 'Total List Price'
# orderBy 'Total Quantity' in descending order

##### 8.4.2. Write the Streaming data to Memory in "Complete" mode

In [47]:
purchase_orders_gold_query = (
    df_fact_pos_products_per_supplier_gold.writeStream \
    # create the new "fact_pos_products_per_supplier" query
    .format("memory") \
    .outputMode("complete") \
    .queryName("fact_pos_products_per_supplier")
    .start()
)

In [48]:
wait_until_stream_is_ready(purchase_orders_gold_query, 1)

The stream has processed 1 batchs


##### 8.4.3. Query the Gold Data from Memory

In [49]:
df_fact_pos_products_per_supplier = spark.sql("SELECT * FROM fact_pos_products_per_supplier")
df_fact_pos_products_per_supplier.printSchema()

root
 |-- vendor_name: string (nullable = true)
 |-- product_Name: string (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- total_cost: double (nullable = true)



##### 8.4.4. Create the Final Selection

In [51]:
df_fact_pos_products_per_supplier_gold_final = df_fact_pos_products_per_supplier \
.select(col("vendor_name"), \
        col("product_name"), \
        col("total_quantity"), \
        col("total_cost"))
# .select() the 'company' column as 'Supplier', the 'product_name' column as 'Product',
# along with the 'Total Quantity', 'Total Unit Cost', and 'Total List Price' columns

##### 8.4.5. Load the Final Results into a New Table and Display the Results

In [52]:
df_fact_pos_products_per_supplier_gold_final.write.saveAsTable(f"{dest_database}.fact_pos_products_per_supplier", mode="overwrite")
spark.sql(f"SELECT * FROM {dest_database}.fact_pos_products_per_supplier").toPandas()

,vendor_name,product_name,total_quantity,total_cost
0,Superior Bicycles,Rear Brakes,27500,5.034267e+06
1,Superior Bicycles,Front Brakes,27500,5.034267e+06
2,Professional Athletic Consultants,ML Road Tire,19800,2.814793e+06
3,Professional Athletic Consultants,LL Road Tire,19800,2.814793e+06
4,Professional Athletic Consultants,HL Road Tire,19250,2.810441e+06
...,...,...,...,...
401,WestAmerica Bicycle Co.,Thin-Jam Hex Nut 15,33,8.387494e+03
402,Advanced Bicycles,Thin-Jam Hex Nut 12,36,8.380324e+03
403,Pro Sport Industries,External Lock Washer 8,33,8.372213e+03
404,Pro Sport Industries,External Lock Washer 2,33,8.372213e+03


### 9.0. Stop the Spark Session

In [53]:
spark.stop()